In [ ]:

# ============================================================
# MINIGDP: INDONESIA INFLATION FORECASTING
# Machine Learning - Random Forest Regression
# ============================================================

# STEP 1: Install libraries
!pip install pandas_datareader scikit-learn matplotlib --quiet

# STEP 2: Import libraries
import pandas as pd
import numpy as np
from pandas_datareader import wb
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_absolute_error
import matplotlib.pyplot as plt

# STEP 3: Download World Bank data
indicators = {
    'FP.CPI.TOTL.ZG': 'CPI_inflation',
    'FM.LBL.BMNY.ZG': 'M2_growth'
}

data = wb.download(
    indicator=list(indicators.keys()),
    country='IDN',
    start=2000,
    end=2023
)

data = data.reset_index()
data = data.rename(columns=indicators)

data = data.pivot_table(
    index='year',
    values=list(indicators.values()),
    aggfunc='first'
)

data = data.sort_index()

print("\n=== SAMPLE DATA ===")
print(data.head())

# STEP 4: Feature and Target
# Predict next year's inflation using current M2 growth

data['target'] = data['CPI_inflation'].shift(-1)

df = data.dropna()

X = df[['M2_growth']]
y = df['target']

# STEP 5: Time Series Cross Validation
tscv = TimeSeriesSplit(n_splits=5)
maes = []

for train_idx, test_idx in tscv.split(X):

    X_train = X.iloc[train_idx]
    X_test = X.iloc[test_idx]

    y_train = y.iloc[train_idx]
    y_test = y.iloc[test_idx]

    model = RandomForestRegressor(
        n_estimators=100,
        max_depth=3,
        random_state=42
    )

    model.fit(X_train, y_train)

    predictions = model.predict(X_test)

    mae = mean_absolute_error(y_test, predictions)
    maes.append(mae)

    print(f"Fold MAE: {mae:.4f}")

print("\nAverage MAE:", round(np.mean(maes), 4))

# STEP 6: Train Final Model
final_model = RandomForestRegressor(
    n_estimators=100,
    max_depth=3,
    random_state=42
)

final_model.fit(X, y)

# STEP 7: Predict next year's inflation
latest_m2 = data[['M2_growth']].dropna().iloc[[-1]]

forecast = final_model.predict(latest_m2)[0]

print("\n=== INFLATION FORECAST ===")
print("Latest M2 growth:", round(latest_m2.iloc[0, 0], 2), "%")
print("Predicted next-year inflation:", round(forecast, 2), "%")

# STEP 8: Historical Predictions
historical_predictions = final_model.predict(X)

results = pd.DataFrame({
    'Actual': y,
    'Predicted': historical_predictions
}, index=y.index)

# STEP 9: Visualization
plt.figure(figsize=(12, 5))

plt.plot(
    results.index,
    results['Actual'],
    marker='o',
    label='Actual Inflation'
)

plt.plot(
    results.index,
    results['Predicted'],
    marker='x',
    linestyle='--',
    label='Predicted Inflation'
)

plt.title('Indonesia Inflation: Actual vs Predicted')
plt.xlabel('Year')
plt.ylabel('Inflation (%)')
plt.legend()
plt.grid(True)
plt.show()

print("\n=== MODEL RESULTS ===")
print(results.round(2))
